[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/22_anomaly_detection/02_isolation_forest_lof_ocsvm.ipynb)

# Bài 02 — Isolation Forest, LOF, One-Class SVM

**🎯 Mục tiêu**
- **Isolation Forest**: bất thường dễ bị "cô lập" bằng vài phép cắt ngẫu nhiên — cài đặt cây cô lập from scratch.
- **Local Outlier Factor**: so sánh mật độ của điểm với mật độ láng giềng — bắt bất thường **cục bộ**.
- **One-Class SVM**: học ranh giới bao quanh dữ liệu bình thường (kernel).
- So sánh trên các cấu trúc dữ liệu khác nhau; tham số `contamination`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/22_anomaly_detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ý tưởng Isolation Forest

Chia không gian bằng các **phép cắt ngẫu nhiên** (chọn feature ngẫu nhiên, ngưỡng ngẫu nhiên trong khoảng giá trị). Điểm bất thường — ít và khác biệt — bị **cô lập sau rất ít phép cắt**; điểm bình thường nằm giữa đám đông cần nhiều phép cắt. Độ dài đường đi trung bình qua nhiều cây ngẫu nhiên → điểm bất thường.

In [ ]:
def isolation_path_length(x, X, depth=0, max_depth=20, r=None):
    """Số phép cắt ngẫu nhiên cần để cô lập x khỏi tập X."""
    if len(X) <= 1 or depth >= max_depth:
        return depth + c_factor(len(X))
    j = r.integers(X.shape[1])
    lo, hi = X[:, j].min(), X[:, j].max()
    if lo == hi:
        return depth + c_factor(len(X))
    t = r.uniform(lo, hi)
    side = X[:, j] <= t
    same = side if x[j] <= t else ~side
    return isolation_path_length(x, X[same], depth + 1, max_depth, r)


def c_factor(n):
    """Độ dài đường đi trung bình của tìm kiếm không thành công trong BST — dùng để chuẩn hoá."""
    if n <= 1:
        return 0.0
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n


X = np.vstack([rng.normal(0, 1, (256, 2)), [[4, 4], [-3.5, 3.5]]])
r = np.random.default_rng(1)
paths = np.array([[isolation_path_length(x, X[r.choice(len(X), 128, replace=False)], r=r) for _ in range(100)] for x in X])
mean_path = paths.mean(1)
score = 2 ** (-mean_path / c_factor(128))                       # điểm bất thường ∈ (0, 1]: gần 1 = bất thường
print(f"độ dài đường đi trung bình: điểm bình thường ≈ {mean_path[:256].mean():.1f} | hai điểm lạ: {mean_path[256:].round(1)}")
print(f"anomaly score: bình thường ≈ {score[:256].mean():.2f} | hai điểm lạ: {score[256:].round(2)}")
sk_scores = -IsolationForest(n_estimators=100, max_samples=128, random_state=0).fit(X).score_samples(X)
print(f"sklearn IsolationForest: bình thường ≈ {sk_scores[:256].mean():.2f} | hai điểm lạ: {sk_scores[256:].round(2)}")

Đặc điểm Isolation Forest: không cần khoảng cách hay mật độ (nhanh, $O(n\log n)$), mỗi cây chỉ dùng mẫu con nhỏ (`max_samples` = 256 mặc định), hoạt động tốt với nhiều chiều; yếu với bất thường **cục bộ** nằm gần cụm dày.

## 2. Local Outlier Factor

LOF so sánh **mật độ cục bộ** của điểm với mật độ của các láng giềng: $\text{LOF}(x) \approx \frac{\text{mật độ trung bình của láng giềng}}{\text{mật độ của } x}$. LOF ≈ 1: như láng giềng; LOF ≫ 1: thưa hơn hẳn láng giềng → bất thường **so với vùng của nó**.

Ví dụ: một cụm rất dày và một cụm thưa; điểm A nằm ngay cạnh cụm dày (khoảng cách tuyệt đối nhỏ nhưng **bất thường so với cụm dày**):

In [ ]:
rl = np.random.default_rng(0)
dense = rl.normal([0, 0], 0.15, (300, 2))
sparse = rl.normal([4, 4], 1.5, (200, 2))
A = np.array([[0.7, 0.7]])
Xl = np.vstack([dense, sparse, A])
lof = LocalOutlierFactor(n_neighbors=20).fit(Xl)
lof_score = -lof.negative_outlier_factor_
iso = -IsolationForest(random_state=0).fit(Xl).score_samples(Xl)
for name, s in [("LOF", lof_score), ("Isolation Forest", iso)]:
    rank = (s[:-1] > s[-1]).sum()
    print(f"{name:17s}: điểm A xếp hạng bất thường thứ {rank + 1}/{len(Xl)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, s) in zip(axes, [("LOF", lof_score), ("Isolation Forest", iso)]):
    sc = ax.scatter(*Xl.T, c=s, cmap="Reds", s=12)
    ax.scatter(*A.T, s=250, facecolors="none", edgecolors="b", lw=2, label="điểm A")
    plt.colorbar(sc, ax=ax, label="điểm bất thường")
    ax.set_title(name)
    ax.legend()
plt.show()

LOF xếp A bất thường nhất (thưa hơn hẳn so với cụm dày bên cạnh); Isolation Forest xếp hơn hai chục điểm rìa của cụm thưa lên trước A — những điểm này xa đám đông về khoảng cách tuyệt đối, nhưng **bình thường so với cụm thưa** của chúng.

## 3. One-Class SVM

Tìm ranh giới (kernel RBF) bao quanh phần lớn dữ liệu train, tham số $\nu$ ≈ tỉ lệ điểm được phép nằm ngoài. Nhạy với scaling và `gamma`; chậm khi dữ liệu lớn ($O(n^2)$).

In [ ]:
Xo = rng.normal(0, 1, (400, 2))
Xo = np.vstack([Xo[Xo[:, 0] < 1.2], rng.normal([0, 0], 1, (50, 2)) + [[3, 0]]])   # dữ liệu bình thường hình "lưỡi liềm"
ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma=0.5).fit(Xo)
gx, gy = np.meshgrid(np.linspace(-4, 6, 200), np.linspace(-4, 4, 200))
Zg = ocsvm.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
plt.contourf(gx, gy, Zg, levels=20, cmap="RdBu", alpha=0.5)
plt.contour(gx, gy, Zg, levels=[0], colors="k")
plt.scatter(*Xo.T, s=8)
plt.title(f"One-Class SVM (ν = 0.05): {(ocsvm.predict(Xo) == -1).mean():.1%} điểm train nằm ngoài ranh giới")
plt.show()

## 4. So sánh trên ba cấu trúc dữ liệu

In [ ]:
def scenario(kind, seed):
    r = np.random.default_rng(seed)
    if kind == "một cụm Gaussian":
        normal = r.normal(0, 1, (500, 2))
    elif kind == "nhiều cụm, mật độ khác nhau":
        normal = np.vstack([r.normal([0, 0], 0.3, (250, 2)), r.normal([4, 4], 1.2, (250, 2))])
    else:                                                       # "đa tạp cong"
        t = r.uniform(0, np.pi, 500)
        normal = np.c_[3 * np.cos(t), 3 * np.sin(t)] + r.normal(0, 0.2, (500, 2))
    anomalies = r.uniform(normal.min(0) - 1, normal.max(0) + 1, (25, 2))
    return np.vstack([normal, anomalies]), np.r_[np.zeros(500), np.ones(25)]


detectors = {
    "Isolation Forest": lambda X: -IsolationForest(random_state=0).fit(X).score_samples(X),
    "LOF": lambda X: -LocalOutlierFactor(n_neighbors=20).fit(X).negative_outlier_factor_,
    "One-Class SVM": lambda X: -OneClassSVM(nu=0.05, gamma="scale").fit(X).decision_function(X),
    "Mahalanobis": lambda X: np.einsum("ij,jk,ik->i", X - X.mean(0), np.linalg.inv(np.cov(X.T)), X - X.mean(0)),
}
fig, axes = plt.subplots(3, 4, figsize=(22, 15))
for i, kind in enumerate(["một cụm Gaussian", "nhiều cụm, mật độ khác nhau", "đa tạp cong"]):
    Xs, ys = scenario(kind, seed=i)
    Xs = StandardScaler().fit_transform(Xs)
    for j, (name, f) in enumerate(detectors.items()):
        s = f(Xs)
        axes[i, j].scatter(*Xs.T, c=s, cmap="Reds", s=8)
        axes[i, j].scatter(*Xs[ys == 1].T, s=60, facecolors="none", edgecolors="b")
        axes[i, j].set_title(f"{kind} — {name}\nPR-AUC = {average_precision_score(ys, s):.2f}", fontsize=10)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
plt.tight_layout()
plt.show()

- **Mahalanobis** (một Gaussian) ngang các detector khác khi dữ liệu đúng là một cụm, nhưng tụt mạnh khi dữ liệu bình thường **nhiều cụm** hoặc **cong** — một ellipse không bao được hình dạng đó.
- **LOF** tốt nhất trên đa tạp cong (chỉ nhìn láng giềng gần); nhưng với hai cụm mật độ rất khác nhau, các bất thường rơi gần cụm thưa có mật độ "giống" cụm thưa → bị bỏ sót.
- **Isolation Forest**: không phải lúc nào cũng tốt nhất, nhưng **ổn định nhất** qua cả ba cấu trúc — lý do nó là lựa chọn mặc định tốt.
- **One-Class SVM** với tham số mặc định (`gamma="scale"`, ν = 0.05) đứng cuối hoặc gần cuối — nhạy tham số, mà không có nhãn thì khó tinh chỉnh.

> Mỗi kịch bản chỉ có 25 bất thường → PR-AUC dao động đáng kể theo seed; đừng đọc chênh lệch 0.02–0.05 như kết luận chắc chắn. Nhãn ở đây dùng **chỉ để so sánh** — thực tế thường có rất ít nhãn (bài 03).

## 5. Tham số `contamination` và ngưỡng

`contamination` = tỉ lệ bất thường **dự kiến** — chỉ dùng để đặt **ngưỡng** cho `predict` (mặc định "auto"), không ảnh hưởng điểm số `score_samples`. Trong thực tế, nên làm việc với **điểm số** và chọn ngưỡng theo năng lực xử lý (top-k mỗi ngày) hoặc theo validation có nhãn.

In [ ]:
Xs, ys = scenario("một cụm Gaussian", seed=0)
for c in [0.01, 0.05, 0.2]:
    pred = IsolationForest(contamination=c, random_state=0).fit(Xs).predict(Xs) == -1
    print(f"contamination = {c:<5}: cảnh báo {pred.sum():3d} điểm | bắt {pred[ys == 1].sum()}/25 bất thường")

## 🧠 Kiểm tra nhanh

1. Vì sao điểm bất thường có đường đi ngắn trong cây cô lập?
2. LOF = 3 nghĩa là gì?
3. Khi nào LOF tốt hơn Isolation Forest?
4. `contamination` ảnh hưởng gì tới `score_samples`?

## 📌 Tóm tắt

- Isolation Forest: cô lập bằng cắt ngẫu nhiên; nhanh, mặc định tốt.
- LOF: mật độ tương đối so với láng giềng — bất thường cục bộ.
- One-Class SVM: ranh giới kernel; nhạy tham số, chậm khi dữ liệu lớn.
- Không có detector tốt nhất cho mọi cấu trúc; `contamination` chỉ đặt ngưỡng.

## 📚 Tài liệu tham khảo

- Liu, Ting & Zhou (2008) — *Isolation Forest*
- Breunig et al. (2000) — *LOF: Identifying Density-Based Local Outliers*
- Schölkopf et al. (2001) — *Estimating the Support of a High-Dimensional Distribution* (One-Class SVM)